# Stick-figure image batch (SD1.5, T4)
Runtime > Change runtime type > T4 GPU, then Run all. Images go to Google Drive/video_images. Re-running skips finished files.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, os, json
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image
pipe = StableDiffusionPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5', torch_dtype=torch.float16, safety_checker=None).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()
OUT='/content/drive/MyDrive/video_images'; os.makedirs(OUT, exist_ok=True)
STYLE='flat 2D cartoon, simple stick figure character with round head, thick black outlines, plain cream beige background, minimal simple scene, flat colors, no shading, childish doodle style, '
NEG='photo, realistic, 3d, shading, gradient, detailed, painting, text, watermark, signature, blurry, multiple characters, extra limbs'

In [ ]:
# Prompts: one per line in Drive/video_images/prompts.txt (fallback: 5 style tests)
P=OUT+'/prompts.txt'
if os.path.exists(P):
    prompts=[l.strip() for l in open(P) if l.strip()]
else:
    prompts=['a stick figure sitting at a desk looking worried, clock on the wall',
             'a stick figure running away from a giant question mark',
             'two stick figures talking at a table with coffee cups',
             'a stick figure lying in bed awake at night, moon in the window',
             'a stick figure standing on a cliff edge looking at the sunrise']
print(len(prompts),'prompts')

In [ ]:
import time
W,H=768,432; t0=time.time(); n=0
for i,p in enumerate(prompts):
    f=f'{OUT}/img_{i:04d}.png'
    if os.path.exists(f): continue
    g=torch.Generator('cuda').manual_seed(1000+i)
    im=pipe(STYLE+p, negative_prompt=NEG, width=W, height=H, num_inference_steps=22, guidance_scale=7.5, generator=g).images[0]
    im.resize((1280,720), Image.LANCZOS).save(f); n+=1
    if n%25==0: print(i, round(time.time()-t0), 's', flush=True)
print('done', n, 'images in', round(time.time()-t0), 's')